# Query GCP propension de compras

# Librerias

In [3]:
import os
import pandas as pd
import numpy as np
import warnings
from google.cloud import bigquery as bigquery
from datetime import datetime
import datetime as dt
from datetime import date, timedelta
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns',800)
pd.set_option('display.float_format',lambda x:'%.2f'%x)

# Carga de Insumos

## Sabana HANA

In [ ]:
sabana_churn_V4 = 'gs://ent-prd-sandbox-fdo-bucket/ent-prd-sandbox-fdo-bucket/PropensionCompras/SabanasHana/InsumoHana_SabanaAnalitica_PropCompras_01Jul2026_Jun26.parquet'
df_prcom=pd.read_parquet(sabana_churn_V4)
df_prcom.shape

(2826168, 242)

In [5]:
df_prcom['CP']=df_prcom['CP'].astype(float).astype('Int64')
df_prcom['OFICINA_SERVICIO']=df_prcom['OFICINA_SERVICIO'].fillna("0").apply(lambda x:x[-8:])
df_prcom['OFICINA_SERVICIO']=df_prcom['OFICINA_SERVICIO'].astype(float).astype('Int64')

## Vector Potencial (Segmentacion) 

* Revisar el Insumo que se esta usando 
* Dejar el segmento y el score obtenido en VP

### Union por CC

In [6]:
vector = 'gs://ent-prd-sandbox-fdo-bucket/ent-prd-sandbox-fdo-bucket/VectorPotencial/ResultadoModelo/VectorPotencial/SegmentacionVectorPotencial202606.csv'
df_vp = pd.read_csv(vector)
del(df_vp['Clave_Mun'])

df_vp.drop_duplicates(subset=['CC'],inplace=True)
df_vp["CC"]=df_vp["CC"].apply(lambda x: (str(x)[-8:])).astype("int64")

print(df_vp.shape)
df_vp.head(3)

(431, 3)


,CC,VECTOR_POTENCIAL,INDICE_VECTOR
0,30011440,MEDIA,0.55
1,30011213,MEDIA,0.36
2,30021036,MEDIA_BAJA,0.24


In [7]:
df_vp.CC.nunique()

431

In [8]:
seleccion = df_vp.select_dtypes(include = ["Int64"])
seleccion.head(3)


,CC
0,30011440
1,30011213
2,30021036


In [9]:
df_prcom=df_prcom.merge(df_vp[['CC','VECTOR_POTENCIAL','INDICE_VECTOR']], left_on ='OFICINA_SERVICIO',right_on='CC',how='left')
del(df_prcom['CC'])

In [10]:
#El siguiente registro no se encontro pudo unir
df_prcom[(df_prcom['OFICINA_SERVICIO'].isna()==False)&(df_prcom['VECTOR_POTENCIAL'].isna()==True)]['OFICINA_SERVICIO'].unique()

<IntegerArray>
[30031144, 30031133, 30031097, 30031037, 30031099, 30031025, 30031096,
 30031114, 30031073, 30031198,
 ...
 30031049, 30031235, 30031125, 30031050, 30031001, 30071020, 30031176,
 30031107, 30031231, 30071060]
Length: 154, dtype: Int64

In [11]:
#El siguiente registro no se encontro pudo unir
df_prcom[(df_prcom['OFICINA_SERVICIO'].isna()==False)&(df_prcom['INDICE_VECTOR'].isna()==True)][['NUM_CLIENTE','OFICINA_SERVICIO','INDICE_VECTOR']].head()

,NUM_CLIENTE,OFICINA_SERVICIO,INDICE_VECTOR
7,96041668,30031144,NaN
10,84520280,30031133,NaN
11,40130558,30031097,NaN
12,13985160,30031037,NaN
15,84222754,30031099,NaN


In [12]:
print('La mediana es',df_vp['INDICE_VECTOR'].median(axis = 0))
print('El promedio es',df_vp['INDICE_VECTOR'].mean(axis = 0))

La mediana es 0.0817914453005732
El promedio es 0.13713284941143097


In [13]:
#Se remplaza el valor, para que no quede vacio (se realizo un analisi previo)
df_prcom['VECTOR_POTENCIAL'][df_prcom['VECTOR_POTENCIAL'].isna()==True]='Media'

In [14]:
#Corroboramos que ya no tengamos ningun null
df_prcom[(df_prcom['OFICINA_SERVICIO'].isna()==False)&(df_prcom['VECTOR_POTENCIAL'].isna()==True)][['NUM_CLIENTE','OFICINA_SERVICIO','VECTOR_POTENCIAL']]

,NUM_CLIENTE,OFICINA_SERVICIO,VECTOR_POTENCIAL


In [15]:
df_prcom['INDICE_VECTOR'][df_prcom['INDICE_VECTOR'].isna()==True] = df_vp['INDICE_VECTOR'].median(axis = 0)

In [16]:
#Corroboramos que ya no tengamos ningun null
df_prcom[(df_prcom['OFICINA_SERVICIO'].isna()==False)&(df_prcom['INDICE_VECTOR'].isna()==True)][['NUM_CLIENTE','OFICINA_SERVICIO','INDICE_VECTOR']]

,NUM_CLIENTE,OFICINA_SERVICIO,INDICE_VECTOR


## MUNICIPIO

In [17]:
df_cp=pd.read_csv('gs://ent-prd-sandbox-fdo-bucket/ent-prd-sandbox-fdo-bucket/PredictivoRotacion/InsumosEntrenamiento/sepomex.csv')
df_cp['clave_municipio']=df_cp['clave_municipio'].astype(int)

df_cp.rename(columns={'CP_CLIENTE':'CP','clave_municipio':'CVE_MUN'},inplace=True)
df_cp.head(2)

df_cp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35549 entries, 0 to 35548
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   CP       35549 non-null  int64
 1   CVE_MUN  35549 non-null  int64
dtypes: int64(2)
memory usage: 555.6 KB


In [18]:
#reemplazando caracteres 

df_prcom['CP']=df_prcom['CP'].replace({np.nan : 0})
df_prcom['CP']=df_prcom['CP'].astype(int)

In [19]:
df_prcom=df_prcom.merge(df_cp, left_on ='CP',right_on='CP',how='left')
print(df_prcom.shape)
df_prcom.head(3)

(2826168, 245)


,NUM_CLIENTE,BP,NUM_CUENTA_AHORRO,ID_ESTATUS_CUENTA,ANTIGUEDAD_AHORRO,CUENTA_AHORRO,FEC_APERTURA_CUENTA,ESTATUS_CUENTA_DESC,ID_PRODUCTO_AHORRO,PRODUCTO_AHORRO,SALDO_CORTE,ESTATUS_PLASTICO_ID,ESTATUS_PLASTICO_DESC,FECHA_VIGENCIA_PLASTICO,FLAG_AMIFAVOR,FLAG_AHORROS,FLAG_INVERSIONES,FLAG_CRD,NUM_INVERSIONES,SALDO_CORTE_INVER,PLAZO,SALDO_CORTE_AH_M_2,SALDO_CORTE_AH_M_1,SALDO_CORTE_AH_M0,SALDO_CORTE_AH_M1,SALDO_CORTE_AH_M2,SALDO_CORTE_AH_M3,SALDO_CORTE_AH_M4,SALDO_CORTE_AH_M5,SALDO_CORTE_AH_M6,SALDO_CORTE_AH_M7,SALDO_CORTE_AH_M8,SALDO_AH_M_2,SALDO_AH_M_1,SALDO_AH_M0,SALDO_AH_M1,SALDO_AH_M2,SALDO_AH_M3,SALDO_AH_M4,SALDO_AH_M5,SALDO_AH_M6,SALDO_AH_M7,SALDO_AH_M8,SALDO_INV_CORTE_M_2,SALDO_INV_CORTE_M_1,SALDO_INV_CORTE_M0,SALDO_INV_CORTE_M1,SALDO_INV_CORTE_M2,SALDO_INV_CORTE_M3,SALDO_INV_CORTE_M4,SALDO_INV_CORTE_M5,SALDO_INV_CORTE_M6,SALDO_INV_CORTE_M7,SALDO_INV_CORTE_M8,SALDO_INV_M_2,SALDO_INV_M_1,SALDO_INV_M0,SALDO_INV_M1,SALDO_INV_M2,SALDO_INV_M3,SALDO_INV_M4,SALDO_INV_M5,SALDO_INV_M6,SALDO_INV_M7,SALDO_INV_M8,MONTO_TX_IN_M_2,TOT_TX_IN_M_2,MONTO_TX_IN_M_1,TOT_TX_IN_M_1,MONTO_TX_IN_M0,TOT_TX_IN_M0,MONTO_TX_IN_M1,TOT_TX_IN_M1,MONTO_TX_IN_M2,TOT_TX_IN_M2,MONTO_TX_IN_M3,TOT_TX_IN_M3,MONTO_TX_IN_M4,TOT_TX_IN_M4,MONTO_TX_IN_M5,TOT_TX_IN_M5,MONTO_TX_IN_M6,TOT_TX_IN_M6,MONTO_TX_IN_M7,TOT_TX_IN_M7,MONTO_TX_IN_M8,TOT_TX_IN_M8,MONTO_TX_OUT_M_2,TOT_TX_OUT_M_2,MONTO_TX_OUT_M_1,TOT_TX_OUT_M_1,MONTO_TX_OUT_M0,TOT_TX_OUT_M0,MONTO_TX_OUT_M1,TOT_TX_OUT_M1,MONTO_TX_OUT_M2,TOT_TX_OUT_M2,MONTO_TX_OUT_M3,TOT_TX_OUT_M3,MONTO_TX_OUT_M4,TOT_TX_OUT_M4,MONTO_TX_OUT_M5,TOT_TX_OUT_M5,MONTO_TX_OUT_M6,TOT_TX_OUT_M6,MONTO_TX_OUT_M7,TOT_TX_OUT_M7,MONTO_TX_OUT_M8,TOT_TX_OUT_M8,MONTO_SPEI_IN_M_2,TOT_SPEI_TX_IN_M_2,MONTO_SPEI_IN_M_1,TOT_SPEI_TX_IN_M_1,MONTO_SPEI_IN_M0,TOT_SPEI_TX_IN_M0,MONTO_SPEI_IN_M1,TOT_SPEI_TX_IN_M1,MONTO_SPEI_IN_M2,TOT_SPEI_TX_IN_M2,MONTO_SPEI_IN_M3,TOT_SPEI_TX_IN_M3,MONTO_SPEI_IN_M4,TOT_SPEI_TX_IN_M4,MONTO_SPEI_IN_M5,TOT_SPEI_TX_IN_M5,MONTO_SPEI_IN_M6,TOT_SPEI_TX_IN_M6,MONTO_SPEI_IN_M7,TOT_SPEI_TX_IN_M7,MONTO_SPEI_IN_M8,TOT_SPEI_TX_IN_M8,MONTO_SPEI_OUT_M_2,TOT_SPEI_TX_OUT_M_2,MONTO_SPEI_OUT_M_1,TOT_SPEI_TX_OUT_M_1,MONTO_SPEI_OUT_M0,TOT_SPEI_TX_OUT_M0,MONTO_SPEI_OUT_M1,TOT_SPEI_TX_OUT_M1,MONTO_SPEI_OUT_M2,TOT_SPEI_TX_OUT_M2,MONTO_SPEI_OUT_M3,TOT_SPEI_TX_OUT_M3,MONTO_SPEI_OUT_M4,TOT_SPEI_TX_OUT_M4,MONTO_SPEI_OUT_M5,TOT_SPEI_TX_OUT_M5,MONTO_SPEI_OUT_M6,TOT_SPEI_TX_OUT_M6,MONTO_SPEI_OUT_M7,TOT_SPEI_TX_OUT_M7,MONTO_SPEI_OUT_M8,TOT_SPEI_TX_OUT_M8,MONTO_DESEM_CRED_IN_M_2,TOT_DESEM_CRED_TX_IN_M_2,MONTO_DESEM_CRED_IN_M_1,TOT_DESEM_CRED_TX_IN_M_1,MONTO_DESEM_CRED_IN_M0,TOT_DESEM_CRED_TX_IN_M0,MONTO_DESEM_CRED_IN_M1,TOT_DESEM_CRED_TX_IN_M1,MONTO_DESEM_CRED_IN_M2,TOT_DESEM_CRED_TX_IN_M2,MONTO_DESEM_CRED_IN_M3,TOT_DESEM_CRED_TX_IN_M3,MONTO_DESEM_CRED_IN_M4,TOT_DESEM_CRED_TX_IN_M4,MONTO_DESEM_CRED_IN_M5,TOT_DESEM_CRED_TX_IN_M5,MONTO_DESEM_CRED_IN_M6,TOT_DESEM_CRED_TX_IN_M6,MONTO_DESEM_CRED_IN_M7,TOT_DESEM_CRED_TX_IN_M7,MONTO_DESEM_CRED_IN_M8,TOT_DESEM_CRED_TX_IN_M8,MONTO_COMPRAS_OUT_M_2,TOT_COMPRAS_TX_OUT_M_2,MONTO_COMPRAS_OUT_M_1,TOT_COMPRAS_TX_OUT_M_1,MONTO_COMPRAS_OUT_M0,TOT_COMPRAS_TX_OUT_M0,MONTO_COMPRAS_OUT_M1,TOT_COMPRAS_TX_OUT_M1,MONTO_COMPRAS_OUT_M2,TOT_COMPRAS_TX_OUT_M2,MONTO_COMPRAS_OUT_M3,TOT_COMPRAS_TX_OUT_M3,MONTO_COMPRAS_OUT_M4,TOT_COMPRAS_TX_OUT_M4,MONTO_COMPRAS_OUT_M5,TOT_COMPRAS_TX_OUT_M5,MONTO_COMPRAS_OUT_M6,TOT_COMPRAS_TX_OUT_M6,MONTO_COMPRAS_OUT_M7,TOT_COMPRAS_TX_OUT_M7,MONTO_COMPRAS_OUT_M8,TOT_COMPRAS_TX_OUT_M8,TOT_NO_USOTX_M8,TOT_NO_USOTX_M7,TOT_NO_USOTX_M6,TOT_NO_USOTX_M5,TOT_NO_USOTX_M4,TOT_NO_USOTX_M3,TOT_NO_USOTX_M2,TOT_NO_USOTX_M1,TOT_NO_USOTX_M0,TOT_NO_USOTX_M_1,TOT_NO_USOTX_M_2,FECHA_NAC_CLIENTE,EDAD,ID_ESTADO_CIVIL,ESTADO_CIVIL,ESCOLARIDAD_DESC,ESCOLARIDAD,CP,ESTADO_ID,GENERO,ID_GENERO,TIPO_VIVIENDA_ID,TIPO_VIVIENDA,OCUPACION_ID,OCUPACION,DEPENDIENTES,ACTIVIDAD_ECONOMICA_ID,ACTIVIDAD_ECONOMICA,HIJOS,TIPO_INGRESO_ID,TIPO_INGRESO,ID_PRODUCTO,PRODUCTO_CRED_DESC,MONTO_DESEMBOLSO,ID_CANAL_DISPERSION,CANAL_DISPERSION,ID_METODO_DISPERS

## Inclusión Financiera

In [20]:
#inc_fin = 'gs://ent-prd-sandbox-fdo-bucket/ent-prd-sandbox-fdo-bucket/InclusionFinanciera/IIF_140725_1224.txt'
inc_fin = 'gs://ent-prd-sandbox-fdo-bucket/ent-prd-sandbox-fdo-bucket/InclusionFinanciera/IIF_220426_0925.parquet'

dfif = pd.read_parquet(inc_fin) #se agrega la separación ya ue el archivo que se esta leyendo es txt sparado por |
dfif.head(2)

,cve_mun,i_clave_estado,i_region,i_estado,i_municipio,i_pob_total,i_pob_adulta,i_pob_adulta_F,i_pobn_adulta_M,i_tipo_pob,i_grado_rez_social,subi_infraestructura,subi_captacion,subi_credito,IIF,categoria_iif
0,01001,1,Occidente y Bajío,Aguascalientes,Aguascalientes,1029221,784241,407094,377147,Metrópoli,Muy bajo,0.14,0.17,0.19,0.17,Alta
1,01002,1,Occidente y Bajío,Aguascalientes,Asientos,57713,40492,20448,20044,Urbano,Muy bajo,0.03,0.04,0.04,0.04,Baja


In [21]:
dfif.rename(columns={"IIF": "INDICE_INCLUSION",}, inplace=True)
dfif.rename(columns={"cve_mun": "Clave_Mun"}, inplace=True)

In [22]:
dfif = dfif[['INDICE_INCLUSION',"subi_credito","subi_captacion","subi_infraestructura",'Clave_Mun']]
dfif.head(2)

,INDICE_INCLUSION,subi_credito,subi_captacion,subi_infraestructura,Clave_Mun
0,0.17,0.19,0.17,0.14,01001
1,0.04,0.04,0.04,0.03,01002


In [23]:
dfif["Clave_Mun"] = dfif["Clave_Mun"].astype(float)

In [24]:
df_prcom=df_prcom.merge(dfif, left_on ='CVE_MUN',right_on='Clave_Mun',how='left')

In [25]:
df_prcom.head(3)

,NUM_CLIENTE,BP,NUM_CUENTA_AHORRO,ID_ESTATUS_CUENTA,ANTIGUEDAD_AHORRO,CUENTA_AHORRO,FEC_APERTURA_CUENTA,ESTATUS_CUENTA_DESC,ID_PRODUCTO_AHORRO,PRODUCTO_AHORRO,SALDO_CORTE,ESTATUS_PLASTICO_ID,ESTATUS_PLASTICO_DESC,FECHA_VIGENCIA_PLASTICO,FLAG_AMIFAVOR,FLAG_AHORROS,FLAG_INVERSIONES,FLAG_CRD,NUM_INVERSIONES,SALDO_CORTE_INVER,PLAZO,SALDO_CORTE_AH_M_2,SALDO_CORTE_AH_M_1,SALDO_CORTE_AH_M0,SALDO_CORTE_AH_M1,SALDO_CORTE_AH_M2,SALDO_CORTE_AH_M3,SALDO_CORTE_AH_M4,SALDO_CORTE_AH_M5,SALDO_CORTE_AH_M6,SALDO_CORTE_AH_M7,SALDO_CORTE_AH_M8,SALDO_AH_M_2,SALDO_AH_M_1,SALDO_AH_M0,SALDO_AH_M1,SALDO_AH_M2,SALDO_AH_M3,SALDO_AH_M4,SALDO_AH_M5,SALDO_AH_M6,SALDO_AH_M7,SALDO_AH_M8,SALDO_INV_CORTE_M_2,SALDO_INV_CORTE_M_1,SALDO_INV_CORTE_M0,SALDO_INV_CORTE_M1,SALDO_INV_CORTE_M2,SALDO_INV_CORTE_M3,SALDO_INV_CORTE_M4,SALDO_INV_CORTE_M5,SALDO_INV_CORTE_M6,SALDO_INV_CORTE_M7,SALDO_INV_CORTE_M8,SALDO_INV_M_2,SALDO_INV_M_1,SALDO_INV_M0,SALDO_INV_M1,SALDO_INV_M2,SALDO_INV_M3,SALDO_INV_M4,SALDO_INV_M5,SALDO_INV_M6,SALDO_INV_M7,SALDO_INV_M8,MONTO_TX_IN_M_2,TOT_TX_IN_M_2,MONTO_TX_IN_M_1,TOT_TX_IN_M_1,MONTO_TX_IN_M0,TOT_TX_IN_M0,MONTO_TX_IN_M1,TOT_TX_IN_M1,MONTO_TX_IN_M2,TOT_TX_IN_M2,MONTO_TX_IN_M3,TOT_TX_IN_M3,MONTO_TX_IN_M4,TOT_TX_IN_M4,MONTO_TX_IN_M5,TOT_TX_IN_M5,MONTO_TX_IN_M6,TOT_TX_IN_M6,MONTO_TX_IN_M7,TOT_TX_IN_M7,MONTO_TX_IN_M8,TOT_TX_IN_M8,MONTO_TX_OUT_M_2,TOT_TX_OUT_M_2,MONTO_TX_OUT_M_1,TOT_TX_OUT_M_1,MONTO_TX_OUT_M0,TOT_TX_OUT_M0,MONTO_TX_OUT_M1,TOT_TX_OUT_M1,MONTO_TX_OUT_M2,TOT_TX_OUT_M2,MONTO_TX_OUT_M3,TOT_TX_OUT_M3,MONTO_TX_OUT_M4,TOT_TX_OUT_M4,MONTO_TX_OUT_M5,TOT_TX_OUT_M5,MONTO_TX_OUT_M6,TOT_TX_OUT_M6,MONTO_TX_OUT_M7,TOT_TX_OUT_M7,MONTO_TX_OUT_M8,TOT_TX_OUT_M8,MONTO_SPEI_IN_M_2,TOT_SPEI_TX_IN_M_2,MONTO_SPEI_IN_M_1,TOT_SPEI_TX_IN_M_1,MONTO_SPEI_IN_M0,TOT_SPEI_TX_IN_M0,MONTO_SPEI_IN_M1,TOT_SPEI_TX_IN_M1,MONTO_SPEI_IN_M2,TOT_SPEI_TX_IN_M2,MONTO_SPEI_IN_M3,TOT_SPEI_TX_IN_M3,MONTO_SPEI_IN_M4,TOT_SPEI_TX_IN_M4,MONTO_SPEI_IN_M5,TOT_SPEI_TX_IN_M5,MONTO_SPEI_IN_M6,TOT_SPEI_TX_IN_M6,MONTO_SPEI_IN_M7,TOT_SPEI_TX_IN_M7,MONTO_SPEI_IN_M8,TOT_SPEI_TX_IN_M8,MONTO_SPEI_OUT_M_2,TOT_SPEI_TX_OUT_M_2,MONTO_SPEI_OUT_M_1,TOT_SPEI_TX_OUT_M_1,MONTO_SPEI_OUT_M0,TOT_SPEI_TX_OUT_M0,MONTO_SPEI_OUT_M1,TOT_SPEI_TX_OUT_M1,MONTO_SPEI_OUT_M2,TOT_SPEI_TX_OUT_M2,MONTO_SPEI_OUT_M3,TOT_SPEI_TX_OUT_M3,MONTO_SPEI_OUT_M4,TOT_SPEI_TX_OUT_M4,MONTO_SPEI_OUT_M5,TOT_SPEI_TX_OUT_M5,MONTO_SPEI_OUT_M6,TOT_SPEI_TX_OUT_M6,MONTO_SPEI_OUT_M7,TOT_SPEI_TX_OUT_M7,MONTO_SPEI_OUT_M8,TOT_SPEI_TX_OUT_M8,MONTO_DESEM_CRED_IN_M_2,TOT_DESEM_CRED_TX_IN_M_2,MONTO_DESEM_CRED_IN_M_1,TOT_DESEM_CRED_TX_IN_M_1,MONTO_DESEM_CRED_IN_M0,TOT_DESEM_CRED_TX_IN_M0,MONTO_DESEM_CRED_IN_M1,TOT_DESEM_CRED_TX_IN_M1,MONTO_DESEM_CRED_IN_M2,TOT_DESEM_CRED_TX_IN_M2,MONTO_DESEM_CRED_IN_M3,TOT_DESEM_CRED_TX_IN_M3,MONTO_DESEM_CRED_IN_M4,TOT_DESEM_CRED_TX_IN_M4,MONTO_DESEM_CRED_IN_M5,TOT_DESEM_CRED_TX_IN_M5,MONTO_DESEM_CRED_IN_M6,TOT_DESEM_CRED_TX_IN_M6,MONTO_DESEM_CRED_IN_M7,TOT_DESEM_CRED_TX_IN_M7,MONTO_DESEM_CRED_IN_M8,TOT_DESEM_CRED_TX_IN_M8,MONTO_COMPRAS_OUT_M_2,TOT_COMPRAS_TX_OUT_M_2,MONTO_COMPRAS_OUT_M_1,TOT_COMPRAS_TX_OUT_M_1,MONTO_COMPRAS_OUT_M0,TOT_COMPRAS_TX_OUT_M0,MONTO_COMPRAS_OUT_M1,TOT_COMPRAS_TX_OUT_M1,MONTO_COMPRAS_OUT_M2,TOT_COMPRAS_TX_OUT_M2,MONTO_COMPRAS_OUT_M3,TOT_COMPRAS_TX_OUT_M3,MONTO_COMPRAS_OUT_M4,TOT_COMPRAS_TX_OUT_M4,MONTO_COMPRAS_OUT_M5,TOT_COMPRAS_TX_OUT_M5,MONTO_COMPRAS_OUT_M6,TOT_COMPRAS_TX_OUT_M6,MONTO_COMPRAS_OUT_M7,TOT_COMPRAS_TX_OUT_M7,MONTO_COMPRAS_OUT_M8,TOT_COMPRAS_TX_OUT_M8,TOT_NO_USOTX_M8,TOT_NO_USOTX_M7,TOT_NO_USOTX_M6,TOT_NO_USOTX_M5,TOT_NO_USOTX_M4,TOT_NO_USOTX_M3,TOT_NO_USOTX_M2,TOT_NO_USOTX_M1,TOT_NO_USOTX_M0,TOT_NO_USOTX_M_1,TOT_NO_USOTX_M_2,FECHA_NAC_CLIENTE,EDAD,ID_ESTADO_CIVIL,ESTADO_CIVIL,ESCOLARIDAD_DESC,ESCOLARIDAD,CP,ESTADO_ID,GENERO,ID_GENERO,TIPO_VIVIENDA_ID,TIPO_VIVIENDA,OCUPACION_ID,OCUPACION,DEPENDIENTES,ACTIVIDAD_ECONOMICA_ID,ACTIVIDAD_ECONOMICA,HIJOS,TIPO_INGRESO_ID,TIPO_INGRESO,ID_PRODUCTO,PRODUCTO_CRED_DESC,MONTO_DESEMBOLSO,ID_CANAL_DISPERSION,CANAL_DISPERSION,ID_METODO_DISPERS

# Generación de la Sabana Final Propension Compras

In [26]:
# Conexion Big Query
project_id = 'ent-prd-sandbox-mlops'
os.environ['GOOGLE_CLOUD_PROJECT'] = project_id
client = bigquery.Client(project=project_id)

In [27]:
input_fecha='2026-06-30'   ##<---------- ÚLTIMA DÍA DEL MES DE CORTE

In [28]:
querydigital2='''
select
cast(cd_orig_cte as INT64) as BP, 1 AS CTE_DIG3M
FROM `ent-prd-datawarecloud.ent_prd_mud_credito.hch_cnl_digital_agr_mes`
WHERE true
and fecha_periodo<='{}'
and fecha_periodo>=DATE_SUB('{}',interval 3 MONTH)
and id_canal_digital = 1
and id_tipo_transaccion in (5,7,8,10,11,12,23,24,25,26)
and id_tipo_movimiento =1
qualify ROW_NUMBER() OVER(PARTITION BY cd_orig_cte ORDER BY fecha_periodo) = 1
'''

In [29]:
dfcliendig= client.query(querydigital2.format(input_fecha,input_fecha)).to_dataframe()
print(dfcliendig.shape)
dfcliendig.head()

(719849, 2)


,BP,CTE_DIG3M
0,29027,1
1,37721,1
2,69148,1
3,70777,1
4,73061,1


In [32]:
df_prcom_1=df_prcom.iloc[:1000000]
df_prcom_2=df_prcom.iloc[1000000:]

In [33]:
df_prcom_1=df_prcom_1.merge(dfcliendig,how="left", left_on="BP",right_on="BP")
df_prcom_2=df_prcom_2.merge(dfcliendig,how="left", left_on="BP",right_on="BP")


In [34]:
#print(df_churn.shape)
#df_churn.sample(3)
df_prcom_1.shape

(1000000, 251)

In [35]:
df_prcom_2.shape

(1826168, 251)

In [36]:
#/home/jupyter/ent-sandbox-fdo-jupyter-repository/Churn (abandono)/
#ent-prd-sandbox-fdo-bucket/ent-prd-sandbox-fdo-bucket/Churn/SabanasConDatosExternos
ruta_salida_sabana = 'gs://ent-prd-sandbox-fdo-bucket/ent-prd-sandbox-fdo-bucket/PropensionCompras/SabanasAnaliticas/SabanaAnalitica_PropCompras_20260701_Jun26_1.parquet'
df_prcom_1.to_parquet(ruta_salida_sabana)

In [37]:
ruta_salida_sabana = 'gs://ent-prd-sandbox-fdo-bucket/ent-prd-sandbox-fdo-bucket/PropensionCompras/SabanasAnaliticas/SabanaAnalitica_PropCompras_20260701_Jun26_2.parquet'
df_prcom_2.to_parquet(ruta_salida_sabana)

# Generación de la Sabana Transacciones digitales

## CARGA INSUMOS

In [38]:
# Conexion Big Query
project_id = 'ent-prd-sandbox-mlops'
os.environ['GOOGLE_CLOUD_PROJECT'] = project_id
client = bigquery.Client(project=project_id)

In [39]:
input_fecha='2026-06-30'   ##<---------- ÚLTIMA DÍA DEL MES DE CORTE

In [40]:
querydigital='''
SELECT  CD_ORIG_CTE AS ID_CTE,FORMAT_DATE("%Y%m",HCH.FECHA_PERIODO) AS mes_periodo, 
                  COUNT(monto_transaccion) AS NUM_DIG_TX,
                  SUM(monto_transaccion) AS MON_DIG_TX
          FROM `ent-prd-datawarecloud.ent_prd_mud_credito.hch_cnl_digital_det_dia` HCH
          WHERE 
           HCH.FECHA_PERIODO <= LAST_DAY('{}') AND 
           HCH.FECHA_PERIODO >= DATE_SUB('{}',interval 8 MONTH)
          AND ID_CANAL_DIGITAL = 1
          AND id_tipo_transaccion IN (23,7,10,11,8,5,25,12,8,24,2,26)
          AND HCH.CD_ORIG_CTE IN (SELECT DISTINCT CD_ORIG_CTE
                                  FROM `ent-prd-datawarecloud.ent_prd_mud_credito.hch_cnl_digital_det_dia` HCH
                                  WHERE HCH.ID_TIPO_TRANSACCION = 14 --ACTIVACION DE SERVICIOS BM
                                  )
          GROUP BY CD_ORIG_CTE,mes_periodo
'''

In [41]:
dftransdig= client.query(querydigital.format(input_fecha,input_fecha)).to_dataframe()
print(dftransdig.shape)
dftransdig.head()

(4397637, 4)


,ID_CTE,mes_periodo,NUM_DIG_TX,MON_DIG_TX
0,0080086366,202510,3,1170.000000000
1,0069513075,202510,1,10.000000000
2,0010760344,202510,3,2913.000000000
3,0200716557,202510,4,10158.000000000
4,0088170938,202510,2,1388.000000000


In [42]:
dftransdig["MON_DIG_TX"]=dftransdig["MON_DIG_TX"].astype("float")

In [43]:
aux=pd.DataFrame(dftransdig["mes_periodo"].drop_duplicates().sort_values(ascending=True))

In [44]:
aux=aux.reset_index()

In [45]:
aux=aux.drop(columns="index")

In [46]:
aux

,mes_periodo
0,202510
1,202511
2,202512
3,202601
4,202602
5,202603
6,202604
7,202605
8,202606


In [47]:
#aux["orden"]=["T-8","T-7","T-6","T-5","T-4","T-3","T-2","T-1","T0","T1","T2"]
aux["orden"]=["T-8","T-7","T-6","T-5","T-4","T-3","T-2","T-1","T0"]

In [48]:
dic=dict(zip(aux["mes_periodo"],aux["orden"]))

In [49]:
dftransdig["mes_periodo"]=dftransdig["mes_periodo"].map(dic)

In [50]:
aux2=pd.pivot_table(dftransdig,index="ID_CTE",columns="mes_periodo",aggfunc="sum", fill_value=0).reset_index()
    

In [51]:
aux2.columns=aux2.columns.map(lambda x: '_'.join(x))

In [52]:
aux2

,ID_CTE_,MON_DIG_TX_T-1,MON_DIG_TX_T-2,MON_DIG_TX_T-3,MON_DIG_TX_T-4,MON_DIG_TX_T-5,MON_DIG_TX_T-6,MON_DIG_TX_T-7,MON_DIG_TX_T-8,MON_DIG_TX_T0,NUM_DIG_TX_T-1,NUM_DIG_TX_T-2,NUM_DIG_TX_T-3,NUM_DIG_TX_T-4,NUM_DIG_TX_T-5,NUM_DIG_TX_T-6,NUM_DIG_TX_T-7,NUM_DIG_TX_T-8,NUM_DIG_TX_T0
0,0000020347,0.00,0.00,0.00,0.00,44407.00,19510.00,0.00,0.00,0.00,0,0,0,0,11,2,0,0,0
1,0000020861,0.00,0.00,30.00,0.00,0.00,0.00,0.00,0.00,0.00,0,0,1,0,0,0,0,0,0
2,0000020922,0.00,0.00,0.00,0.00,0.00,39201.00,0.00,0.00,0.00,0,0,0,0,0,2,0,0,0
3,0000020926,3300.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1,0,0,0,0,0,0,0,0
4,0000021221,0.00,0.00,2977.00,0.00,21782.22,13620.00,3077.00,0.00,0.00,0,0,2,0,9,4,2,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1145867,0201527051,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,10000.00,0,0,0,0,0,0,0,0,2
1145868,0201529075,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,5741.00,0,0,0,0,0,0,0,0,1
1145869,0201529301,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,150.00,0,0,0,0,0,0,0,0,1
1145870,0201530454,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,10.00,0,0,0,0,0,0,0,0,1


In [53]:
ruta_salida_sabana = 'gs://ent-prd-sandbox-fdo-bucket/ent-prd-sandbox-fdo-bucket/PropensionCompras/SabanasAnaliticas/SabanaTransacciones_PropCompras_20260701_Jun26.parquet'
aux2.to_parquet(ruta_salida_sabana)